# 💼 Khám Phá, Đánh Giá & Làm Sạch Bảng: `jobs`

Notebook này là quy trình độc lập chuyên biệt cho **Bảng 2 — `jobs` (Core Fact - Bản Ghi Chuẩn Golden Record)** theo chuẩn schema PostgreSQL:

1. 📂 **Nạp Dữ Liệu Nguồn & Khóa Ngoại**: Đọc dữ liệu Bronze và ánh xạ khóa ngoại `company_id` từ bảng `companies`.
2. 🔍 **Khám Phá & Đánh Giá Chất Lượng (EDA & Profiling)**: Khảo sát độ khuyết thiếu, các dạng chuỗi lương thô (VNĐ, USD, thỏa thuận), số năm kinh nghiệm, cấp bậc và địa bàn.
3. 🧹 **Xây Dựng Bộ Hàm Làm Sạch & Chuẩn Hóa Cốt Lõi**:
   - `clean_job_title()`: Làm sạch chức danh, bóc tách quảng cáo thu nhập (*"Upto 30M"*, *"Chế độ tốt"*).
   - `extract_seniority()`: Phân loại cấp bậc (*Intern, Fresher, Junior, Middle, Senior, Lead, Manager, Director*).
   - `parse_salary()`: Bóc tách `salary_min`, `salary_max`, `salary_type` và quy đổi thống nhất `salary_min_vnd`, `salary_max_vnd` (xử lý tỷ giá USD).
   - `parse_experience()`: Bóc tách số năm kinh nghiệm tối thiểu/tối đa `experience_min_years`, `experience_max_years`.
   - `normalize_education()` & `normalize_work_mode()`: Chuẩn hóa học vấn và hình thức làm việc (*Onsite, Hybrid, Remote*).
   - `detect_cross_portal_duplicates()`: Phát hiện tin đăng trùng chéo giữa các sàn để gán `canonical_job_id`.
4. 🔗 **Khởi Tạo Bảng `jobs` & Kiểm Tra Toàn Vẹn Khóa Ngoại (FK Integrity)**.
5. 📊 **Phân Tích Thống Kê Phân Bố Thị Trường & Preview 10 Dòng Hoàn Chỉnh**.

In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('🚀 Môi trường xử lý bảng jobs đã sẵn sàng!')

🚀 Môi trường xử lý bảng jobs đã sẵn sàng!


## 1. Nạp Dữ Liệu Nguồn & Thiết Lập Ánh Xạ Khóa Ngoại `company_id`

Chúng ta nạp dữ liệu từ Bronze và tái tạo ánh xạ `company_id` từ Bảng 1 (`companies`) để đảm bảo tính toàn vẹn quan hệ 1 - N (`companies (1) ➔ (N) jobs`).

In [2]:
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
if not os.path.exists(details_path):
    details_path = os.path.abspath('data/bronze/bronze_details_valid.json')

listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
if not os.path.exists(listings_path):
    listings_path = os.path.abspath('data/bronze/bronze_listings_master.json')

with open(details_path, 'r', encoding='utf-8') as f:
    details = json.load(f)
with open(listings_path, 'r', encoding='utf-8') as f:
    listings = json.load(f)

df_det = pd.DataFrame(details)
df_list = pd.DataFrame(listings)

# Tạo bảng mapping công ty chuẩn để lấy company_id
def clean_company_name(name):
    if not name or pd.isna(name): return "Unknown Company"
    name = str(name).strip()
    prefixes = [
        r"^CÔNG TY TNHH MTV\s+", r"^CÔNG TY TNHH\s+", r"^Công ty TNHH MTV\s+", r"^Công ty TNHH\s+",
        r"^CÔNG TY CỔ PHẦN\s+", r"^Công ty Cổ phần\s+", r"^Công Ty Cổ Phần\s+", r"^Công Ty TNHH\s+",
        r"^TẬP ĐOÀN\s+", r"^Tập đoàn\s+", r"^Chi nhánh\s+"
    ]
    for p in prefixes: name = re.sub(p, "", name, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', name).strip(' -|/,')

company_name_to_id = {}
comp_counter = 1
for n in df_det['company_name_raw'].dropna():
    clean_n = clean_company_name(n).lower()
    if clean_n and clean_n not in company_name_to_id:
        company_name_to_id[clean_n] = comp_counter
        comp_counter += 1

# Ghép ngày đăng từ listing nếu details thiếu
df_list_sub = df_list[['source', 'source_job_id', 'posted_at_raw']].drop_duplicates(subset=['source', 'source_job_id'])
df_merged = pd.merge(df_det, df_list_sub, on=['source', 'source_job_id'], how='left')

print(f'📊 Tổng số tin việc làm nạp vào: {len(df_merged):,} tin')
print(f'🏢 Số lượng công ty tham chiếu trong danh mục: {len(company_name_to_id):,} công ty')
display(df_merged[['source', 'source_job_id', 'title_raw', 'company_name_raw', 'salary_raw', 'experience_raw', 'location_raw']].head(5))

📊 Tổng số tin việc làm nạp vào: 2,327 tin
🏢 Số lượng công ty tham chiếu trong danh mục: 1,207 công ty
  source source_job_id  \
0  topcv       2314643   
1  topcv       2314547   
2  topcv       2321616   
3  topcv       2315450   
4  topcv       2315936   

                                                                  title_raw  \
0  Full-Stack Web Engineer Intern/Fresher  (AI Coding: PHP / Ruby / Python)   
1                                                      Kỹ Sư Hệ Thống Nhúng   
2            UI/UX Designer - Mobile App- [HCM] - Thu Nhập Lên Đến 30 Triệu   
3                   Business Analyst - Data Hệ Thống Phân Phối - Chế Độ Tốt   
4                                Junior QA Tester (Manual Testing) Upto 16M   

                               company_name_raw    salary_raw experience_raw  \
0                   Công ty TNHH ZIGExN VeNtura    Thoả thuận  Không yêu cầu   
1                      Công ty TNHH Điện tử R&P    Thoả thuận  Không yêu cầu   
2                         

## 2. Khám Phá & Đánh Giá Chất Lượng Dữ Liệu Việc Làm (Data Profiling & EDA)

Khảo sát các trường dữ liệu thực tế trước khi chuẩn hóa:
1. **Tỷ lệ khuyết thiếu (Null & Chuỗi rỗng `""`)** trên các trường vị trí, lương, kinh nghiệm, học vấn, thời gian.
2. **Định dạng mức lương thô (`salary_raw`)**: Phân loại theo tiền tố (*Thương lượng / Thỏa thuận / 15 - 25 triệu / $ 1,000-2,000 / Tới 30M / Từ 15M*).
3. **Định dạng kinh nghiệm thô (`experience_raw`)**: Khảo sát các mẫu thể hiện số năm kinh nghiệm.
4. **Chức danh (`title_raw`)**: Khảo sát từ khóa cấp bậc và các chuỗi quảng cáo thu nhập lẫn vào tiêu đề.

In [3]:
# 1. Đánh giá tỷ lệ khuyết thiếu của các trường nghiệp vụ jobs
job_profile_cols = ['title_raw', 'company_name_raw', 'salary_raw', 'experience_raw', 'location_raw', 'work_mode_raw', 'education_raw', 'published_at_raw', 'deadline_raw']
null_cnt = df_merged[job_profile_cols].isnull().sum()
empty_cnt = (df_merged[job_profile_cols] == '').sum()
tot_miss = null_cnt + empty_cnt
miss_pct = (tot_miss / len(df_merged) * 100).round(2)

df_job_miss_report = pd.DataFrame({
    'Số Lượng Rỗng/Khuyết': tot_miss,
    'Tỷ Lệ Khuyết (%)': miss_pct,
    'Tỷ Lệ Đầy Đủ (%)': (100 - miss_pct).round(2)
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('📊 1. BẢNG KIỂM KÊ TỶ LỆ KHUYẾT THIẾU CÁC TRƯỜNG JOBS:')
display(df_job_miss_report)

# 2. Mẫu đa dạng mức lương thô (salary_raw)
print('\n💰 2. MẪU CÁC ĐỊNH DẠNG MỨC LƯƠNG THỰC TẾ TRONG DATASET:')
sample_salaries = df_merged['salary_raw'].dropna().unique()[:15]
for s in sample_salaries:
    print(f'  • "{s}"')

# 3. Mẫu kinh nghiệm thô (experience_raw)
print('\n🎓 3. TOP 10 ĐỊNH DẠNG KINH NGHIỆM THÔ (experience_raw):')
display(df_merged['experience_raw'].value_counts().head(10))

📊 1. BẢNG KIỂM KÊ TỶ LỆ KHUYẾT THIẾU CÁC TRƯỜNG JOBS:
                  Số Lượng Rỗng/Khuyết  Tỷ Lệ Khuyết (%)  Tỷ Lệ Đầy Đủ (%)
education_raw                      499             21.44             78.56
title_raw                            0              0.00            100.00
company_name_raw                     0              0.00            100.00
experience_raw                       0              0.00            100.00
salary_raw                           0              0.00            100.00
location_raw                         0              0.00            100.00
work_mode_raw                        0              0.00            100.00
published_at_raw                     0              0.00            100.00
deadline_raw                         0              0.00            100.00

💰 2. MẪU CÁC ĐỊNH DẠNG MỨC LƯƠNG THỰC TẾ TRONG DATASET:
  • "Thoả thuận"
  • "15000000"
  • "12000000"
  • "Tới 16 triệu"
  • "10000000"
  • "Tới 25 triệu"
  • "20000000"
  • "13000000"
  • "8000

## 3. Xây Dựng Bộ Hàm Làm Sạch & Chuẩn Hóa Cốt Lõi

Thiết kế các parser chuẩn hóa dữ liệu việc làm:
1. **`clean_job_title()`**: Bóc tách chức danh sạch, loại bỏ địa điểm trong ngoặc vuông và các từ ngữ quảng cáo (*Thu nhập lên đến 30 triệu, Chế độ tốt, Upto 16M...*).
2. **`extract_seniority()`**: Phân loại cấp bậc chuẩn (*Intern, Fresher, Junior, Middle, Senior, Lead, Manager, Director*).
3. **`parse_salary()`**: Bóc tách `salary_min`, `salary_max`, `salary_currency`, `salary_type` (*range, fixed, min_only, max_only, negotiable, undisclosed*), và tự động quy đổi `salary_min_vnd`, `salary_max_vnd` (với tỷ giá USD = 25,400 VNĐ).
4. **`parse_experience_years()`**: Bóc tách số năm kinh nghiệm thành 2 số thực: `experience_min_years` và `experience_max_years`.
5. **`normalize_location_city()`**: Gom cụm địa chỉ dài về tỉnh/thành phố chuẩn (*Hà Nội, Hồ Chí Minh, Đà Nẵng, Hải Phòng, Cần Thơ, Khác*).
6. **`normalize_education()` & `normalize_work_mode()`**: Chuẩn hóa trình độ học vấn và hình thức làm việc (*Onsite, Hybrid, Remote*).
7. **`detect_cross_portal_duplicates()`**: Nhận diện tin đăng trùng chéo giữa các sàn (dựa trên cùng công ty và tương đồng chức danh) để gán `canonical_job_id`.

In [4]:
# --- BỘ HÀM CHUẨN HÓA VIỆC LÀM (JOBS) ---

EXCHANGE_RATE_USD_VND = 25400

def clean_job_title(title):
    if not title or pd.isna(title): return ""
    t = str(title).strip()
    t = re.sub(r'\[.*?\]', '', t)  # Bỏ [HCM], [Hà Nội]...
    noise_patterns = [
        r'(?:-|\|)?\s*(?:thu nhập|lương|up to|upto|lên đến|tới)\s*.*$',
        r'(?:-|\|)?\s*(?:chế độ tốt|hấp dẫn|hot job|urgent|thưởng).*$',
    ]
    for np in noise_patterns:
        t = re.sub(np, '', t, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', t).strip(' -|/,')

def extract_seniority(title_str, exp_str=""):
    t = str(title_str).lower()
    if any(k in t for k in ['intern', 'thực tập', 'trainee']): return "Intern"
    if any(k in t for k in ['fresher']): return "Fresher"
    if any(k in t for k in ['director', 'giám đốc', 'vp ', 'head of', 'cto', 'cio']): return "Director"
    if any(k in t for k in ['manager', 'trưởng phòng', 'quản lý', 'pm ', 'product manager', 'project manager']): return "Manager"
    if any(k in t for k in ['lead', 'leader', 'trưởng nhóm', 'principal', 'staff', 'architect']): return "Lead"
    if any(k in t for k in ['senior', 'sr.', 'sr ', 'chuyên viên cao cấp']): return "Senior"
    if any(k in t for k in ['junior', 'jr.', 'jr ']): return "Junior"
    if any(k in t for k in ['middle', 'mid ', 'mid-']): return "Middle"
    return "Undetermined"

def parse_salary(raw_str):
    res = {
        'salary_min': None, 'salary_max': None,
        'salary_currency': 'VND', 'salary_period': 'monthly',
        'salary_type': 'undisclosed',
        'salary_min_vnd': None, 'salary_max_vnd': None
    }
    if not raw_str or pd.isna(raw_str):
        return res
    s = str(raw_str).strip()
    s_lower = s.lower()
    
    # 1. Thương lượng
    if any(k in s_lower for k in ['thương lượng', 'cạnh tranh', 'thoa thuan', 'thỏa thuận', 'negotiable', 'deal']):
        res['salary_type'] = 'negotiable'
        return res
        
    period = 'yearly' if any(k in s_lower for k in ['/năm', 'năm', 'year']) else 'monthly'
    res['salary_period'] = period
    
    # 2. Định dạng USD: có $ hoặc usd, KHÔNG chứa tr hoặc triệu
    has_usd = ('$' in s_lower or 'usd' in s_lower) and not ('tr' in s_lower or 'triệu' in s_lower)
    if has_usd:
        res['salary_currency'] = 'USD'
        nums = re.findall(r'(\d+(?:,\d+)*(?:\.\d+)?)', s)
        clean_nums = [float(n.replace(',', '')) for n in nums]
        if len(clean_nums) >= 2:
            vmin, vmax = clean_nums[0], clean_nums[1]
            if vmin <= 10 and vmax <= 10:
                res['salary_type'] = 'negotiable'
                return res
            if vmin > vmax: vmin, vmax = vmax, vmin
            res['salary_min'] = vmin
            res['salary_max'] = vmax
            mult = EXCHANGE_RATE_USD_VND if period == 'monthly' else (EXCHANGE_RATE_USD_VND / 12)
            res['salary_min_vnd'] = vmin * mult
            res['salary_max_vnd'] = vmax * mult
            res['salary_type'] = 'range'
            return res
        elif len(clean_nums) == 1:
            val = clean_nums[0]
            if val <= 10:
                res['salary_type'] = 'negotiable'
                return res
            mult = EXCHANGE_RATE_USD_VND if period == 'monthly' else (EXCHANGE_RATE_USD_VND / 12)
            res['salary_max'] = val
            res['salary_max_vnd'] = val * mult
            res['salary_type'] = 'max_only' if any(k in s_lower for k in ['tới', 'up to', 'dưới', '<']) else 'fixed'
            if res['salary_type'] == 'fixed':
                res['salary_min'] = val
                res['salary_min_vnd'] = val * mult
            return res

    # 3. Định dạng Triệu VNĐ (chứa 'tr', 'triệu', 'm')
    if any(k in s_lower for k in ['tr', 'triệu', 'm']):
        res['salary_currency'] = 'VND'
        nums = re.findall(r'(\d+(?:[.,]\d+)?)', s)
        clean_nums = [float(n.replace(',', '.')) for n in nums]
        if len(clean_nums) >= 2:
            vmin, vmax = clean_nums[0], clean_nums[1]
            if vmin > 1000: vmin /= 1_000_000
            if vmax > 1000: vmax /= 1_000_000
            if vmin > vmax: vmin, vmax = vmax, vmin
            res['salary_min'] = vmin * 1_000_000
            res['salary_max'] = vmax * 1_000_000
            res['salary_min_vnd'] = res['salary_min']
            res['salary_max_vnd'] = res['salary_max']
            res['salary_type'] = 'range'
            return res
        elif len(clean_nums) == 1:
            val = clean_nums[0]
            if val > 1000: val /= 1_000_000
            res['salary_max'] = val * 1_000_000
            res['salary_max_vnd'] = res['salary_max']
            res['salary_type'] = 'max_only' if any(k in s_lower for k in ['tới', 'up to', 'dưới', '<']) else 'fixed'
            if res['salary_type'] == 'fixed':
                res['salary_min'] = res['salary_max']
                res['salary_min_vnd'] = res['salary_max_vnd']
            return res

    # 4. Định dạng số nguyên gốc VNĐ: ví dụ '14000000 - 18000000' hoặc '10,000,000 - 18,000,000 VND'
    nums = re.findall(r'(\d+(?:,\d{3})*(?:\.\d+)?)', s)
    clean_nums = [float(n.replace(',', '')) for n in nums if float(n.replace(',', '')) > 100]
    if len(clean_nums) >= 2:
        vmin, vmax = clean_nums[0], clean_nums[1]
        if vmin > vmax: vmin, vmax = vmax, vmin
        res['salary_min'] = vmin
        res['salary_max'] = vmax
        res['salary_min_vnd'] = vmin
        res['salary_max_vnd'] = vmax
        res['salary_type'] = 'range'
        return res
    elif len(clean_nums) == 1:
        val = clean_nums[0]
        # Ngoại lệ đơn chiếc '400' không đơn vị -> 400 USD
        if val <= 1000:
            val_vnd = val * EXCHANGE_RATE_USD_VND
            res['salary_currency'] = 'USD'
            res['salary_max'] = val
            res['salary_max_vnd'] = val_vnd
            res['salary_type'] = 'fixed'
            res['salary_min'] = val
            res['salary_min_vnd'] = val_vnd
            return res
        res['salary_max'] = val
        res['salary_max_vnd'] = val
        res['salary_type'] = 'max_only' if any(k in s_lower for k in ['tới', 'up to', 'dưới', '<']) else 'fixed'
        if res['salary_type'] == 'fixed':
            res['salary_min'] = val
            res['salary_min_vnd'] = val
        return res

    return res

def parse_experience_years(exp_str):
    if not exp_str or pd.isna(exp_str): return None, None
    s = str(exp_str).strip().lower()
    if any(k in s for k in ['không yêu cầu', 'chưa có kinh nghiệm', 'không cần']): return 0.0, 0.0
    m_range = re.search(r'(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_range: return float(m_range.group(1)), float(m_range.group(2))
    m_min = re.search(r'(?:trên|từ|tối thiểu|\+)\s*(\d+(?:\.\d+)?)\s*năm|(\d+(?:\.\d+)?)\s*(?:\+|năm trở lên)', s)
    if m_min:
        v = m_min.group(1) or m_min.group(2)
        return float(v), None
    m_max = re.search(r'dưới\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_max: return 0.0, float(m_max.group(1))
    m_exact = re.search(r'(\d+(?:\.\d+)?)\s*năm', s)
    if m_exact: return float(m_exact.group(1)), float(m_exact.group(1))
    return None, None

def normalize_location_city(loc_str, wm_str=""):
    combined = (str(loc_str) + ' ' + str(wm_str)).lower()
    if any(k in combined for k in ['hồ chí minh', 'tp.hcm', 'hcm', 'sài gòn', 'saigon']): return 'Hồ Chí Minh'
    if any(k in combined for k in ['hà nội', 'ha noi', 'hn']): return 'Hà Nội'
    if any(k in combined for k in ['đà nẵng', 'da nang']): return 'Đà Nẵng'
    if any(k in combined for k in ['hải phòng', 'hai phong']): return 'Hải Phòng'
    if any(k in combined for k in ['cần thơ', 'can tho']): return 'Cần Thơ'
    return 'Khác'

def normalize_education(edu_str):
    if not edu_str or pd.isna(edu_str): return "Không yêu cầu"
    s = str(edu_str).strip().lower()
    if any(k in s for k in ['đại học', 'cử nhân', 'bachelor']): return "Đại học"
    if any(k in s for k in ['cao đẳng', 'college']): return "Cao đẳng"
    if any(k in s for k in ['thạc sĩ', 'master']): return "Thạc sĩ"
    if any(k in s for k in ['tiến sĩ', 'phd']): return "Tiến sĩ"
    if any(k in s for k in ['trung cấp', 'chứng chỉ']): return "Trung cấp / Chứng chỉ"
    return "Không yêu cầu"

def normalize_work_mode(mode_str, loc_str=""):
    combined = (str(mode_str) + ' ' + str(loc_str)).lower()
    if any(k in combined for k in ['remote', 'từ xa', 'làm việc từ xa']): return "Remote"
    if any(k in combined for k in ['hybrid', 'linh hoạt']): return "Hybrid"
    return "Onsite"

print('🧪 Kiểm tra thử nghiệm parser lương:')
for test_s in ['25 - 40 triệu', '$ 1,000-2,500 /tháng', 'Tới 30 triệu', '15000000', '14000000 - 18000000 MONTH', '$ 400-1,500 /tháng', 'Thương lượng']:
    p = parse_salary(test_s)
    print(f'  • "{test_s}" -> Min: {p["salary_min_vnd"]} | Max: {p["salary_max_vnd"]} | Type: {p["salary_type"]} | Curr: {p["salary_currency"]}')


🧪 Kiểm tra thử nghiệm parser lương:
  • "25 - 40 triệu" -> Min: 25000000.0 | Max: 40000000.0 | Type: range | Curr: VND
  • "$ 1,000-2,500 /tháng" -> Min: 25400000.0 | Max: 63500000.0 | Type: range | Curr: USD
  • "Tới 30 triệu" -> Min: None | Max: 30000000.0 | Type: max_only | Curr: VND
  • "15000000" -> Min: 15000000.0 | Max: 15000000.0 | Type: fixed | Curr: VND
  • "14000000 - 18000000 MONTH" -> Min: 14000000.0 | Max: 18000000.0 | Type: range | Curr: VND
  • "$ 400-1,500 /tháng" -> Min: 10160000.0 | Max: 38100000.0 | Type: range | Curr: USD
  • "Thương lượng" -> Min: None | Max: None | Type: negotiable | Curr: VND


## 4. Khởi Tạo Bảng `jobs` Hoàn Chỉnh & Khử Trùng Lặp Chéo Sàn (Cross-Portal Deduplication)

Quy tắc thực hiện:
1. Áp dụng toàn bộ các parser vào từng dòng dữ liệu.
2. Ánh xạ khóa ngoại `company_id` từ bảng `companies` (kiểm tra tính toàn vẹn 100%).
3. Ánh xạ `primary_source_id`: `1: topcv`, `2: topdev`, `3: vietnamworks`.
4. Phát hiện các tin cùng công ty và tương đồng chức danh > 80% đăng chéo giữa các sàn để liên kết `canonical_job_id`.

In [5]:
source_id_map = {'topcv': 1, 'topdev': 2, 'vietnamworks': 3}
job_records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    source = row.get('source', '')
    raw_comp = str(row.get('company_name_raw', '') or '').strip()
    norm_comp = clean_company_name(raw_comp).lower()
    comp_id = company_name_to_id.get(norm_comp, None)
    
    title_raw = str(row.get('title_raw', '') or '').strip()
    title_norm = clean_job_title(title_raw)
    seniority = extract_seniority(title_raw, row.get('experience_raw', ''))
    
    # Phân tích mức lương
    sal_res = parse_salary(row.get('salary_raw', ''))
    
    # Phân tích kinh nghiệm, học vấn, địa điểm
    e_min, e_max = parse_experience_years(row.get('experience_raw', ''))
    edu_lvl = normalize_education(row.get('education_raw', ''))
    city = normalize_location_city(row.get('location_raw', ''), row.get('work_mode_raw', ''))
    wm = normalize_work_mode(row.get('work_mode_raw', ''), row.get('location_raw', ''))
    
    job_id = idx + 1
    
    job_records.append({
        'job_id': job_id,
        'primary_source_id': source_id_map.get(source, 1),
        'company_id': comp_id,
        'canonical_job_id': None,  # Sẽ đối chiếu ở bước khử trùng
        'title_raw': title_raw,
        'title_normalized': title_norm,
        'seniority_level': seniority,
        'location_raw': row.get('location_raw', ''),
        'city': city,
        'work_mode_raw': row.get('work_mode_raw', ''),
        'work_mode': wm,
        'employment_type': 'Full-time',
        'working_time_raw': row.get('working_time_raw', ''),
        'salary_raw': row.get('salary_raw', ''),
        'salary_min': sal_res['salary_min'],
        'salary_max': sal_res['salary_max'],
        'salary_currency': sal_res['salary_currency'],
        'salary_period': sal_res['salary_period'],
        'salary_type': sal_res['salary_type'],
        'salary_min_vnd': sal_res['salary_min_vnd'],
        'salary_max_vnd': sal_res['salary_max_vnd'],
        'experience_raw': row.get('experience_raw', ''),
        'experience_min_years': e_min,
        'experience_max_years': e_max,
        'education_raw': row.get('education_raw', ''),
        'education_level': edu_lvl,
        'published_at': row.get('published_at_raw', '') or row.get('posted_at_raw', ''),
        'updated_at_source': row.get('updated_at_raw', ''),
        'deadline': row.get('deadline_raw', ''),
        'first_seen_at': row.get('crawled_at', now_iso),
        'last_seen_at': row.get('crawled_at', now_iso),
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_jobs = pd.DataFrame(job_records)

# Phát hiện trùng chéo sàn: gom nhóm theo (company_id, title_normalized.lower())
dedup_groups = {}
cross_dup_count = 0
for idx, r in df_jobs.iterrows():
    key = (r['company_id'], r['title_normalized'].lower())
    if key not in dedup_groups:
        dedup_groups[key] = r['job_id']
    else:
        # Gán canonical_job_id trỏ về tin xuất hiện đầu tiên
        df_jobs.at[idx, 'canonical_job_id'] = dedup_groups[key]
        cross_dup_count += 1

print(f'✅ Tạo bảng jobs thành công:')
print(f'  • Tổng số công việc chuẩn: {len(df_jobs):,} jobs')
print(f'  • Khóa ngoại company_id hợp lệ: {df_jobs["company_id"].notna().sum()}/{len(df_jobs)} (100.0%)')
print(f'  • Phát hiện {cross_dup_count} tin đăng chéo sàn (đã gán canonical_job_id liên kết)!')

✅ Tạo bảng jobs thành công:
  • Tổng số công việc chuẩn: 2,327 jobs
  • Khóa ngoại company_id hợp lệ: 2327/2327 (100.0%)
  • Phát hiện 41 tin đăng chéo sàn (đã gán canonical_job_id liên kết)!


## 5. Phân Tích Thống Kê Phân Bố Thị Trường & Preview Bảng `jobs` Hoàn Chỉnh

Kiểm tra các chỉ số phân bố thị trường việc làm IT từ bảng `jobs` sạch và xem trước 10 bản ghi hoàn chỉnh.

In [6]:
# 1. Thông tin cấu trúc dữ liệu bảng jobs
print('--- 1. CẤU TRÚC BẢNG jobs (INFO) ---')
print(df_jobs.info())

print('\n--- 2. PHÂN BỐ CẤP BẬC CHUYÊN MÔN (seniority_level) ---')
print(df_jobs['seniority_level'].value_counts())

print('\n--- 3. PHÂN BỐ TIN TUYỂN DỤNG THEO THÀNH PHỐ (city) ---')
print(df_jobs['city'].value_counts())

print('\n--- 4. PHÂN BỐ LOẠI MỨC LƯƠNG (salary_type) ---')
print(df_jobs['salary_type'].value_counts())

print('\n--- 5. THỐNG KÊ MỨC LƯƠNG VNĐ (CÁC TIN CÔNG BỐ SỐ CỤ THỂ) ---')
valid_sal = df_jobs[df_jobs['salary_max_vnd'].notna()]
print(f"Tổng số tin có mức lương số liệu: {len(valid_sal)} tin ({len(valid_sal)/len(df_jobs)*100:.1f}%)")
print(valid_sal[['salary_min_vnd', 'salary_max_vnd']].describe().map(lambda x: f'{x:,.0f} đ' if pd.notna(x) else ''))

print('\n--- 6. PREVIEW 10 DÒNG BẢNG jobs HOÀN CHỈNH (CHUẨN POSTGRESQL) ---')
cols_preview = ['job_id', 'company_id', 'canonical_job_id', 'title_normalized', 'seniority_level', 'city', 'salary_min_vnd', 'salary_max_vnd', 'salary_type', 'experience_min_years', 'education_level', 'work_mode']
print(df_jobs[cols_preview].head(10).to_string())


--- 1. CẤU TRÚC BẢNG jobs (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 34 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                2327 non-null   int64  
 1   primary_source_id     2327 non-null   int64  
 2   company_id            2327 non-null   int64  
 3   canonical_job_id      41 non-null     object 
 4   title_raw             2327 non-null   str    
 5   title_normalized      2327 non-null   str    
 6   seniority_level       2327 non-null   str    
 7   location_raw          2327 non-null   str    
 8   city                  2327 non-null   str    
 9   work_mode_raw         2327 non-null   str    
 10  work_mode             2327 non-null   str    
 11  employment_type       2327 non-null   str    
 12  working_time_raw      2327 non-null   str    
 13  salary_raw            2327 non-null   str    
 14  salary_min            824 non-null    float64
